In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, CLIPProcessor, CLIPVisionModel
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.utils import resample
from tqdm import tqdm
import copy

# ==========================================
# 1. CONFIGURATION (UPDATE THESE PATHS!)
# ==========================================
class Config:
    # --- CSV FILES ---
    # The file with columns: Image_name, ocr-text, L1_Label, L2_Label
    TRAIN_CSV = "/content/Malayalam_Dataset_Training.csv"

    # The file provided by organizers (meme_id, Level 1, Level 2)
    TEST_CSV = "/content/Malayalam_Test.csv"

    # --- IMAGE FOLDERS ---
    # IMPORTANT: Update these to the actual paths in your Colab file manager
    TRAIN_IMAGE_DIR = "/content/Train_Images" # Folder containing training images
    TEST_IMAGE_DIR = "/content/Test_Images" # Folder containing test images

    # --- MODEL SETTINGS ---
    TEXT_MODEL = "xlm-roberta-base"
    IMAGE_MODEL = "openai/clip-vit-base-patch32"
    MAX_LEN = 128
    BATCH_SIZE = 16
    EPOCHS = 5 # Start with 5, increase if needed
    LR = 2e-5
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 2. DATASET CLASS
# ==========================================
class MalayalamMemeDataset(Dataset):
    def __init__(self, df, tokenizer, processor, img_dir, max_len, is_test=False):
        self.df = df
        self.tokenizer = tokenizer
        self.processor = processor
        self.img_dir = img_dir
        self.max_len = max_len
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # --- 1. Text Processing ---
        # If test set has no OCR, we use empty string
        text = str(row['ocr-text']) if 'ocr-text' in row and pd.notna(row['ocr-text']) else ""

        text_inputs = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )

        # --- 2. Image Processing ---
        img_name = str(row['Image_name']).strip()
        img_path = os.path.join(self.img_dir, img_name)

        try:
            image = Image.open(img_path).convert("RGB")
            image_inputs = self.processor(images=image, return_tensors="pt")
            pixel_values = image_inputs['pixel_values'].squeeze(0)
        except Exception as e:
            # If training, we want to know if images are missing!
            if not self.is_test:
                print(f"⚠️ MISSING IMAGE: {img_path}")
            # Return black image to prevent crash
            pixel_values = torch.zeros((3, 224, 224))

        # --- 3. Labels ---
        if not self.is_test:
            l1 = torch.tensor(int(row['L1_Label']), dtype=torch.long)
            l2 = torch.tensor(int(row['L2_Label']), dtype=torch.long)
        else:
            # Dummy labels for test set
            l1 = torch.tensor(-1, dtype=torch.long)
            l2 = torch.tensor(-1, dtype=torch.long)

        return {
            'input_ids': text_inputs['input_ids'].squeeze(0),
            'attention_mask': text_inputs['attention_mask'].squeeze(0),
            'pixel_values': pixel_values,
            'l1': l1,
            'l2': l2,
            'meme_id': row.get('meme_id', row.get('Image_name', ''))
        }

# ==========================================
# 3. DATA BALANCING (Oversampling)
# ==========================================
def balance_dataframe(df):
    """
    Duplicates minority class rows so the model sees them more often.
    """
    # Create a combined column to stratify (e.g., "0_2")
    df['stratify_col'] = df['L1_Label'].astype(str) + "_" + df['L2_Label'].astype(str)

    # Find the size of the largest group
    max_size = df['stratify_col'].value_counts().max()

    df_balanced = pd.DataFrame()
    for group in df['stratify_col'].unique():
        df_group = df[df['stratify_col'] == group]
        # Resample minority groups to match max_size
        df_upsampled = resample(df_group,
                                replace=True,
                                n_samples=max_size,
                                random_state=42)
        df_balanced = pd.concat([df_balanced, df_upsampled])

    return df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# ==========================================
# 4. MODEL ARCHITECTURE
# ==========================================
class MultimodalMemeClassifier(nn.Module):
    def __init__(self, text_model_name, image_model_name, num_l2_classes=3):
        super().__init__()
        # Text Encoder
        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        self.text_hidden_size = self.text_encoder.config.hidden_size

        # Image Encoder
        self.image_encoder = CLIPVisionModel.from_pretrained(image_model_name)
        self.image_hidden_size = self.image_encoder.config.hidden_size

        # Fusion
        self.fusion = nn.Linear(self.text_hidden_size + self.image_hidden_size, 512)
        self.dropout = nn.Dropout(0.3)
        self.relu = nn.ReLU()

        # Output Heads
        self.l1_classifier = nn.Linear(512, 2)
        self.l2_classifier = nn.Linear(512, num_l2_classes)

    def forward(self, input_ids, attention_mask, pixel_values):
        # Text
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_emb = text_out.pooler_output

        # Image
        image_out = self.image_encoder(pixel_values=pixel_values)
        image_emb = image_out.pooler_output

        # Concatenate & Fuse
        combined = torch.cat((text_emb, image_emb), dim=1)
        fused = self.dropout(self.relu(self.fusion(combined)))

        # Predict
        l1_logits = self.l1_classifier(fused)
        l2_logits = self.l2_classifier(fused)

        return l1_logits, l2_logits

# ==========================================
# 5. TRAINING LOOP
# ==========================================
# ==========================================
# 5. UPDATED TRAINING LOOP (With Detailed Metrics)
# ==========================================
def train_model():
    print(f"🚀 Loading Data from {Config.TRAIN_CSV}...")
    df = pd.read_csv(Config.TRAIN_CSV)

    # 1. SPLIT FIRST
    train_df, val_df = train_test_split(df, test_size=0.15, stratify=df['L1_Label'], random_state=42)

    # 2. BALANCE TRAIN ONLY
    print("⚖️ Balancing Training Data...")
    train_df = balance_dataframe(train_df)

    print(f"Train Size: {len(train_df)} | Val Size: {len(val_df)}")

    # Setup
    tokenizer = AutoTokenizer.from_pretrained(Config.TEXT_MODEL)
    processor = CLIPProcessor.from_pretrained(Config.IMAGE_MODEL)

    train_ds = MalayalamMemeDataset(train_df, tokenizer, processor, Config.TRAIN_IMAGE_DIR, Config.MAX_LEN)
    val_ds = MalayalamMemeDataset(val_df, tokenizer, processor, Config.TRAIN_IMAGE_DIR, Config.MAX_LEN)

    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE)

    model = MultimodalMemeClassifier(Config.TEXT_MODEL, Config.IMAGE_MODEL).to(Config.DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LR)
    criterion = nn.CrossEntropyLoss()

    best_f1 = 0

    # Loop
    for epoch in range(Config.EPOCHS):
        model.train()
        total_loss = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{Config.EPOCHS}"):
            input_ids = batch['input_ids'].to(Config.DEVICE)
            mask = batch['attention_mask'].to(Config.DEVICE)
            pixels = batch['pixel_values'].to(Config.DEVICE)
            l1_target = batch['l1'].to(Config.DEVICE)
            l2_target = batch['l2'].to(Config.DEVICE)

            optimizer.zero_grad()
            l1_logits, l2_logits = model(input_ids, mask, pixels)

            loss = criterion(l1_logits, l1_target) + criterion(l2_logits, l2_target)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        # --- VALIDATION (Detailed) ---
        model.eval()
        val_preds_l1, val_targets_l1 = [], []
        val_preds_l2, val_targets_l2 = [], []

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(Config.DEVICE)
                mask = batch['attention_mask'].to(Config.DEVICE)
                pixels = batch['pixel_values'].to(Config.DEVICE)

                l1_logits, l2_logits = model(input_ids, mask, pixels)

                # Store L1 Predictions
                val_preds_l1.extend(torch.argmax(l1_logits, dim=1).cpu().numpy())
                val_targets_l1.extend(batch['l1'].numpy())

                # Store L2 Predictions
                val_preds_l2.extend(torch.argmax(l2_logits, dim=1).cpu().numpy())
                val_targets_l2.extend(batch['l2'].numpy())

        # Calculate Macro F1 for Model Saving (based on L1)
        val_f1 = f1_score(val_targets_l1, val_preds_l1, average='macro')
        print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f} | Val F1 (L1): {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), "best_model.pth")
            print("💾 Model Saved!")

            # --- PRINT DETAILED REPORT FOR BEST EPOCH ---
            print("\n📊 Classification Report (Level 1 - Troll vs Support):")
            print(classification_report(val_targets_l1, val_preds_l1, target_names=["Troll", "Support"]))

            print("\n📊 Classification Report (Level 2 - Category):")
            # 0=Person, 1=Party, 2=Intersection
            print(classification_report(val_targets_l2, val_preds_l2, target_names=["Person", "Party", "Intersection"]))

    print("\n✅ Training Complete. Best F1:", best_f1)

# ==========================================
# 6. SUBMISSION GENERATION
# ==========================================
def generate_submission():
    print("\n🔮 Generating Submission for Test Data...")

    # 1. Load Test IDs
    test_ids_df = pd.read_csv(Config.TEST_CSV)
    id_col = 'meme_id' if 'meme_id' in test_ids_df.columns else test_ids_df.columns[0]

    # Prepare DataFrame
    test_df = pd.DataFrame()
    # Assume image file is "ID.jpg"
    test_df['Image_name'] = test_ids_df[id_col].astype(str) + ".jpg"
    test_df['meme_id'] = test_ids_df[id_col]

    # 2. Load Model
    model = MultimodalMemeClassifier(Config.TEXT_MODEL, Config.IMAGE_MODEL).to(Config.DEVICE)
    model.load_state_dict(torch.load("best_model.pth"))
    model.eval()

    tokenizer = AutoTokenizer.from_pretrained(Config.TEXT_MODEL)
    processor = CLIPProcessor.from_pretrained(Config.IMAGE_MODEL)

    # Note: is_test=True handles missing OCR and labels
    test_ds = MalayalamMemeDataset(test_df, tokenizer, processor, Config.TEST_IMAGE_DIR, Config.MAX_LEN, is_test=True)
    test_loader = DataLoader(test_ds, batch_size=Config.BATCH_SIZE, shuffle=False)

    # 3. Predict
    l1_preds, l2_preds, ids = [], [], []

    with torch.no_grad():
        for batch in tqdm(test_loader, desc="Predicting"):
            input_ids = batch['input_ids'].to(Config.DEVICE)
            mask = batch['attention_mask'].to(Config.DEVICE)
            pixels = batch['pixel_values'].to(Config.DEVICE)

            l1_logits, l2_logits = model(input_ids, mask, pixels)

            l1_preds.extend(torch.argmax(l1_logits, dim=1).cpu().numpy())
            l2_preds.extend(torch.argmax(l2_logits, dim=1).cpu().numpy())
            ids.extend(batch['meme_id'])

    # 4. Map Numbers -> Text Labels
    final_l1, final_l2 = [], []
    for l1, l2 in zip(l1_preds, l2_preds):
        # Level 1 Mapping
        final_l1.append("TROLL/ OPPOSE" if l1 == 0 else "SUPPORT")

        # Level 2 Mapping
        if l2 == 2:
            l2_str = "Intersection"
        elif l2 == 0:
            l2_str = "Against individual person" if l1 == 0 else "Support for individual person"
        elif l2 == 1:
            l2_str = "Against party" if l1 == 0 else "Support for party"
        final_l2.append(l2_str)

    # 5. Save
    sub = pd.DataFrame({'meme_id': ids, 'Level 1': final_l1, 'Level 2': final_l2})
    sub.to_csv("submission.csv", index=False)
    print("✅ Saved submission.csv")

if __name__ == "__main__":
    train_model()
    generate_submission()

🚀 Loading Data from /content/Malayalam_Dataset_Training.csv...
⚖️ Balancing Training Data...
Train Size: 1650 | Val Size: 75


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias          

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias          

✅ Saved submission.csv
